# Cuaderno 7 — `mutate()`, agrupar y resumir, hasta que salgan solos

**Descripción y Visualización de Datos — UAI 2026 — Clase 7**

Hoy no hay ninguna función nueva. Hay tres que ya viste y que el Sprint 2 va a
necesitar en cada celda, así que la clase es para repetirlas hasta que no
tengas que pensarlas:

| | Qué hace |
|---|---|
| `mutate()` | **agrega una columna nueva**, calculada a partir de las que ya hay |
| `group_by()` + `summarise()` | **una fila por grupo**: cuántos son y un promedio o un porcentaje |
| `arrange()` | ordena la tabla que salió |

El cuaderno tiene dos mitades. En la **Parte A** el profesor corre las celdas
en pantalla y tú sigues: no escribes, miras. En la **Parte B** te toca a ti,
con las mismas cuatro líneas y otras columnas. La **Parte C** es con la
encuesta del curso, y la **Parte D** con la base de tu propio grupo.

Antes de partir: **Archivo → Guardar una copia en Drive.**

## Parte 0 — Punto de partida

Nada nuevo. La CEP demora unos segundos en bajar: son 19 MB.

In [ ]:
library(dplyr)

cep <- read.csv("https://raw.githubusercontent.com/naimbro/naimbro.github.io/main/materiales/2026_descripcion_visualizacion_datos/datos/cep_consolidada_1994_2026.csv")

nrow(cep)

In [ ]:
names(cep)

## Parte A — Demostración (mira, no escribas)

### A1. `mutate()`: una columna que no venía en el archivo

`mutate()` no cambia las filas: la tabla sigue teniendo 96.122 personas. Lo que
hace es **agregar una columna al final**, y hay dos formas de llenarla.

**Forma 1: comparar un número.** ¿Quién tiene 60 años o más?

In [ ]:
cep %>%
  mutate(mayor_60 = edad >= 60) %>%
  select(edad, mayor_60) %>%
  head()

Seis filas, y en cada una la columna nueva contestó `TRUE` o `FALSE`. Una
comparación en R siempre contesta eso.

Contémosla completa:

In [ ]:
cep %>%
  mutate(mayor_60 = edad >= 60) %>%
  count(mayor_60)

Tres filas: 67.457 `FALSE`, 24.103 `TRUE` y **4.562 `NA`**. Esas 4.562 personas
no tienen edad en la base (casi la mitad de las encuestas de 1996 a 1998), y R
no inventa: si no sabe la edad, tampoco sabe si es mayor de 60. Ese `NA` va a
reaparecer en un minuto.

**Forma 2: comparar un texto.** La columna `sit_econ_pais` dice cómo ve cada
persona la economía del país, en palabras. Dos de esas palabras son «Mala» y
«Muy mala», y las queremos juntas: para eso está `|`, que significa **o**.

In [ ]:
cep %>%
  mutate(econ_mala = sit_econ_pais == "Mala" | sit_econ_pais == "Muy mala") %>%
  count(econ_mala)

40.298 personas de 96.122 ven mal la economía. Y esta vez no hay fila `NA`:
todo el mundo contestó algo en esa columna.

> Dos formas, una sola idea: **`mutate(nombre_nuevo = cálculo)`**. El cálculo es
> una comparación, con un número o con un texto, y la columna nueva contesta
> `TRUE` o `FALSE`.

### A2. `summarise()`: muchas filas en una sola

`summarise()` hace lo contrario de `mutate()`: **aplasta la tabla** en una sola
fila. Adentro se escribe `nombre = cálculo`, separados por coma, y cada cálculo
convierte una columna entera en un solo número.

In [ ]:
cep %>%
  mutate(econ_mala = sit_econ_pais == "Mala" | sit_econ_pais == "Muy mala") %>%
  summarise(personas = n(),
            edad_promedio = mean(edad, na.rm = TRUE),
            pct_econ_mala = mean(econ_mala) * 100)

Una fila, tres números: **96.122 personas, 46,7 años de edad promedio, 41,9%
ve mal la economía.** Tres cosas que mirar:

- `n()` cuenta las filas. Va sin nada adentro del paréntesis.
- `mean(edad, na.rm = TRUE)`: sin el `na.rm = TRUE` el promedio sale `NA`,
  porque hay 4.562 edades que faltan y el promedio de algo que incluye un «no
  sé» es «no sé». `econ_mala` no lo necesita, porque no tiene `NA`.
- **`mean(econ_mala) * 100` es un porcentaje.** R trata `TRUE` como 1 y
  `FALSE` como 0, así que el promedio de una columna de `TRUE`/`FALSE` es la
  proporción. Ésa es la receta de todo el cuaderno.

### A3. `group_by()`: la misma cuenta, una vez por grupo

`group_by()` **no hace nada solo**: es una instrucción para el `summarise()`
que viene después. Le dice «esto, pero una vez por cada grupo». Una línea más
y la fila única se convierte en una fila por sexo:

In [ ]:
cep %>%
  mutate(econ_mala = sit_econ_pais == "Mala" | sit_econ_pais == "Muy mala") %>%
  group_by(sexo) %>%
  summarise(personas = n(),
            edad_promedio = mean(edad, na.rm = TRUE),
            pct_econ_mala = mean(econ_mala) * 100)

**Mujeres 45,5%, hombres 36,9%.** Ocho puntos y medio de diferencia que el
41,9% de recién escondía por completo. Misma base, mismas funciones, una línea
de diferencia.

Fíjate también en que la tabla dice `# A tibble`: `group_by()` devuelve un
formato distinto, que imprime más ordenado y sólo las diez primeras filas.

Y ahora agrupando por la columna que creamos en A1, la de los mayores de 60:

In [ ]:
cep %>%
  mutate(econ_mala = sit_econ_pais == "Mala" | sit_econ_pais == "Muy mala",
         mayor_60 = edad >= 60) %>%
  group_by(mayor_60) %>%
  summarise(personas = n(),
            edad_promedio = mean(edad, na.rm = TRUE),
            pct_econ_mala = mean(econ_mala) * 100)

Dos cosas que ver acá. Primero, **un solo `mutate()` creó dos columnas**,
separadas por coma. Segundo, apareció una tercera fila, **`NA`, con 4.562
personas**: `group_by()` trata a los que no tienen edad como un grupo más, y su
`edad_promedio` sale `NaN`. No es un error, es la base diciéndote que ese grupo
existe. En un indicador publicado esa fila se filtra o se declara; nunca se
ignora.

Y el dato: los mayores de 60 ven peor la economía (45,4%) que el resto (41,0%).

### A4. La receta completa

Todo junto, con `arrange()` al final para ordenar. La pregunta: **¿en qué año
Chile vio peor su economía?**

In [ ]:
cep %>%
  mutate(econ_mala = sit_econ_pais == "Mala" | sit_econ_pais == "Muy mala") %>%
  group_by(anio) %>%
  summarise(personas = n(),
            pct_econ_mala = mean(econ_mala) * 100) %>%
  arrange(desc(pct_econ_mala))

**1999 con 63,4%** —la crisis asiática— y pegado, **2022 con 63,3%**. Al otro
extremo, 2010 con 27,1%. Treinta y dos años de ánimo económico en una tabla que
se lee en diez segundos.

Cuatro líneas, y cada una hace una sola cosa:

| Línea | Qué hace |
|---|---|
| `mutate()` | crea la columna `TRUE`/`FALSE` que te interesa |
| `group_by()` | parte la tabla en grupos |
| `summarise()` | cuenta cuántos son (`n()`) y promedia los `TRUE` |
| `arrange(desc())` | ordena de mayor a menor |

Ésa es la receta. Ahora te toca a ti.

## Parte B — Ejercicios (ahora escribes tú)

Cada ejercicio es una de las celdas de arriba con una columna distinta. Si uno
no sale, copia la celda parecida de la Parte A y cámbiale una palabra: así se
empieza. Los ejercicios 1 a 6 son los de hoy; 7 y 8 son para quien va rápido.

**1.** Crea con `mutate()` una columna `joven` que sea `TRUE` cuando `edad` es
menor que 30, y muéstrala junto a `edad` con `select()` y `head()`.

In [ ]:
# Tu código acá

**2.** Crea una columna `media_completa` que sea `TRUE` cuando
`anios_escolaridad` es 12 o más, y cuéntala con `count()`. ¿Cuántas personas
quedaron en `TRUE`, cuántas en `FALSE` y cuántas en `NA`?

In [ ]:
# Tu código acá

**3.** Un solo número para toda la base: **¿qué porcentaje de las personas
encuestadas son mujeres?** Necesitas un `mutate()` que compare `sexo` con
`"Mujer"` y un `summarise()` con `n()` y el promedio por 100.

In [ ]:
# Tu código acá

**4.** La misma cuenta, pero **por año**, ordenada de mayor a menor. ¿En qué
año la muestra tuvo más mujeres y en cuál menos? Mira también la columna
`personas`.

*(La respuesta importa para el Sprint 2: una base que cambia de composición
entre un año y otro no se compara sin decirlo.)*

In [ ]:
# Tu código acá

**5.** Agrupa por `gse` (nivel socioeconómico) y pide dos cosas: `personas` con
`n()` y `edad_promedio` con `mean()`. Acuérdate del `na.rm = TRUE`.

Mira la primera fila de la tabla con atención: hay un grupo que no tiene
nombre. ¿Cuántas personas tiene?

In [ ]:
# Tu código acá

**6.** La receta completa. La columna `chile_hoy` dice si la persona cree que
Chile está «Progresando», «Estancado» o «En decadencia». Calcula el porcentaje
que dice **`"Progresando"`** por año y ordénalo **de menor a mayor** con
`arrange()` sin `desc()`.

El peor año va a salir con **0,0%**. Antes de creerle, corre esto y decide si
ese cero se publica:

In [ ]:
# Tu código acá

In [ ]:
cep %>%
  filter(anio == 2005) %>%
  count(chile_hoy)

**7.** *(Si vas rápido.)* Por región, y sólo de 2022 en adelante: ¿en qué región
hay más gente que nombra la **`"Salud"`** como principal problema del país
(`problema_1`)? Necesitas un `filter(anio >= 2022)` como primera línea, y no
te olvides del `n()`: la región que encabeza el ranking tiene pocos casos, y la
Metropolitana tiene miles.

In [ ]:
# Tu código acá

**8.** *(Si vas rápido.)* El control que salva el Sprint 2. Cuenta por año
cuántas personas tienen dato en `anios_escolaridad`, con
`sum(!is.na(anios_escolaridad))`, y ordena por año de más reciente a más
antiguo. ¿Desde qué año está vacía la columna? ¿Qué le habría pasado al
ejercicio 2 si lo hubieras hecho sólo con datos recientes?

In [ ]:
# Tu código acá

## Parte C — Limpiar y comparar, en la misma cadena

Volvemos a la encuesta del curso, que es chica y sucia: exactamente lo que se
van a encontrar en la base de su proyecto. En la clase 5 descubrimos que
`minutos_viaje` es texto porque alguien escribió `10 min`.

In [ ]:
curso <- read.csv("https://raw.githubusercontent.com/naimbro/naimbro.github.io/main/materiales/2026_descripcion_visualizacion_datos/datos/encuesta_curso.csv")

class(curso$minutos_viaje)

**9.** ¿Cuántos minutos se demora en promedio cada grupo de `transporte`? Son
tres líneas después de `curso`: un `mutate()` con `as.numeric()`, un
`group_by(transporte)` y un `summarise()` con `n()` y `mean()`.

Córrelo primero **sin** `na.rm = TRUE` y mira qué grupo sale `NA`. Después
agrégalo.

In [ ]:
# Tu código acá

**10.** Ahora un porcentaje: ¿qué proporción de cada grupo de `transporte` tiene
un viaje **largo**, de más de 60 minutos? El `mutate()` crea dos columnas: la
numérica y la comparación.

In [ ]:
# Tu código acá

## Parte D — Tu proyecto

Lo mismo, con la base de tu grupo. Ésta es la parte del cuaderno que se
convierte en el Sprint 2.

**11.** Carga la base que declararon en el Sprint 1 con `read.csv()` y muestra
`nrow()` y `names()`. Si no carga, deja el error a la vista y escribe abajo qué
intentaste: eso es lo primero que se conversa con el ayudante.

In [ ]:
# Tu base acá

**12.** Un indicador con la receta: un `mutate()` que cree la columna que te
interesa, un `group_by()` por la columna que quieres comparar, y un
`summarise()` con `n()` adentro. Arriba de la celda, en una línea, el titular
que saldría de esa tabla: con un número y una comparación.

In [ ]:
# Tu indicador acá

In [ ]:
# Espacio libre para probar lo que se te ocurra

In [ ]:
# Espacio libre

## Antes de irte

**Archivo → Guardar** (Ctrl+S).

### Lo que repasaste hoy

| Para qué | Cómo se escribe |
|---|---|
| Una columna `TRUE`/`FALSE` desde un número | `mutate(mayor_60 = edad >= 60)` |
| Una columna `TRUE`/`FALSE` desde un texto | `mutate(econ_mala = sit_econ_pais == "Mala" \| sit_econ_pais == "Muy mala")` |
| Dos columnas de una vez | `mutate(a = ..., b = ...)` |
| Muchas filas en una sola | `summarise(personas = n())` |
| Un promedio ignorando los `NA` | `mean(edad, na.rm = TRUE)` |
| Un porcentaje | `mean(econ_mala) * 100` |
| La misma cuenta por grupo | `group_by(sexo) %>% summarise(...)` |
| Ordenar el resultado | `arrange(desc(pct))` |
| Limpiar antes de comparar | `mutate(minutos_num = as.numeric(minutos_viaje))` |

### Las tres ideas

1. **`mutate()` agrega, `summarise()` aplasta.** Una columna nueva con las
   mismas filas, o una fila nueva con todas las columnas resumidas. Nunca las
   dos cosas a la vez.
2. **`group_by()` no hace nada solo.** Es una instrucción para el
   `summarise()` que viene después.
3. **Siempre `n()`.** Un grupo sin nombre con cinco personas, un año con la
   columna vacía y un segmento con nueve casos se ven idénticos en una tabla
   prolija. Sólo el `n()` los delata.